# PR Review Env: GRPO Training (Colab)\nThis notebook runs a minimal TRL + Unsloth GRPO pipeline against the OpenEnv-compatible PR review environment.

## 1) Clone Repo\nReplace with your GitHub repo URL before running.

In [ ]:
REPO_URL = "https://github.com/<your-org-or-user>/pr-review-env.git"\n!git clone {REPO_URL} repo\n%cd repo

## 2) Install Training Dependencies

In [ ]:
!pip install -q -r requirements-train.txt

## 3) Start OpenEnv Server

In [ ]:
import subprocess, time, os\nserver = subprocess.Popen(["python", "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "7860"])\ntime.sleep(5)\nprint("Server PID:", server.pid)

In [ ]:
!curl -s http://127.0.0.1:7860/health

## 4) Run GRPO Training\nUse a small model first to validate the pipeline quickly.

In [ ]:
!python train_grpo.py \
  --env-base-url http://127.0.0.1:7860 \
  --model-name Qwen/Qwen2.5-0.5B-Instruct \
  --num-samples 24 \
  --num-train-epochs 1 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 2 \
  --num-generations 2 \
  --episodes-per-task 1 \
  --max-episode-steps 2 \
  --max-completion-length 220 \
  --max-new-tokens 220 \
  --output-dir artifacts/grpo_colab_run

## 5) Inspect Reward Curve and Before/After Results

In [ ]:
from IPython.display import Image, display, Markdown\ndisplay(Image(filename='artifacts/grpo_colab_run/plots/reward_curve.png'))\ndisplay(Markdown(open('artifacts/grpo_colab_run/logs/before_after.md', 'r', encoding='utf-8').read()))

In [ ]:
import json\nsummary = json.load(open('artifacts/grpo_colab_run/logs/training_summary.json', 'r', encoding='utf-8'))\nsummary